# 10 — Profiling: why vLLM's sampler pass is slow, and the sampler's share of a real step

Notebook 08 measured times with CUDA events. This notebook explains them with profilers.

**Part A — kernel profiler (Nsight Compute, 1 GPU, no model).** Profiles one launch of each statistics kernel on real logits: vLLM's pass with and without per-logit random numbers, with the probability write, our fused top-k pass with and without random numbers (and vLLM's installed kernel if available). Reports memory and compute throughput (% of peak), occupancy and warp stall reasons, to show *what* limits vLLM's pass.

**Part B — noise ablation (1 GPU, no model, no special permissions).** Times the same streaming pass with: no noise / random numbers only / random numbers + the two logarithms of Gumbel-max / the two logarithms without the random-number generator. This separates the cost of the generator from the cost of the logarithms, and works even if Part A's hardware counters are blocked.

**Part C — vLLM end-to-end trace (2+ GPUs, the model).** Starts a vLLM server per configuration with vLLM's built-in PyTorch profiler, profiles a window of real GSM8K traffic, and breaks the GPU time per denoising step into categories (layers' matmuls, MoE, attention, communication, sampler kernels, top-k, hint). This is the sampler's real share of a step, stock vs. ours.

Parts A and B run in `.venv-2` or `.venv-vllm`; Part C needs `.venv-vllm`. See README, "Profiling needs".

In [ ]:
import os, sys, subprocess, json, re, glob, gzip, shutil, time
if 'torch' in sys.modules:
    raise RuntimeError('Restart the kernel first.')
GPU = None      # Parts A/B: None = the GPU with the most free memory
rows = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.free,memory.total', '--format=csv,noheader,nounits'],
                      capture_output=True, text=True, check=True).stdout.strip().splitlines()
if os.environ.get('SLURM_JOB_ID') and os.environ.get('CUDA_VISIBLE_DEVICES'):   # under Slurm: only the GPUs this job was given
    rows = [r for r in rows if r.split(', ')[0] in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
gpu_free = {int(i): (int(f) / 1024, int(t) / 1024) for i, f, t in (r.split(', ') for r in rows)}
if GPU is None:
    GPU = str(max(gpu_free, key=lambda g: gpu_free[g][0]))
OUT = './runs/nb10'
os.makedirs(OUT, exist_ok=True)
BATCH = 8           # sequences for Parts A/B (8 x 256 rows of 262,144 logits = 2.1 GB)
NCU = shutil.which('ncu') or '/usr/local/cuda/bin/ncu'
print('GPU for parts A/B:', GPU, f'({gpu_free[int(GPU)][0]:.1f} GB free) | ncu:', NCU if os.path.exists(NCU) else 'not found')

## Part A — Nsight Compute

Runs `profile_kernels.py` under `ncu`. Only the launches between `cudaProfilerStart` and `cudaProfilerStop` are profiled (warm-up and compilation happen before), one per kernel variant, in a fixed order.

If the output says `ERR_NVGPUCTRPERM`, this machine does not let normal users read GPU performance counters. Ask the admin to enable them (NVIDIA driver option `NVreg_RestrictProfilingToAdminUsers=0`), or skip to Part B, which needs no counters.

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU)
rep = f'{OUT}/kernels'
cmd = [NCU, '--profile-from-start', 'off', '-f', '-o', rep,
       '--section', 'SpeedOfLight', '--section', 'ComputeWorkloadAnalysis', '--section', 'MemoryWorkloadAnalysis',
       '--section', 'Occupancy', '--section', 'WarpStateStats', '--section', 'LaunchStats',
       sys.executable, 'profile_kernels.py', '--batch', str(BATCH), '--out', f'{OUT}/launch_order.json']
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-2500:])
NCU_OK = r.returncode == 0 and os.path.exists(rep + '.ncu-rep') and 'ERR_NVGPUCTRPERM' not in (r.stdout + r.stderr)
print('ncu profile written:', NCU_OK)

In [ ]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
if NCU_OK:
    order = json.load(open(f'{OUT}/launch_order.json'))['order']
    det = subprocess.run([NCU, '--import', rep + '.ncu-rep', '--csv', '--page', 'details'], capture_output=True, text=True).stdout
    open(f'{OUT}/kernels_details.csv', 'w').write(det)
    d = pd.read_csv(f'{OUT}/kernels_details.csv')
    ids = sorted(d['ID'].unique())
    d['variant'] = d['ID'].map({i: (order[k] if k < len(order) else f'launch {i}') for k, i in enumerate(ids)})
    want = ['Duration', 'DRAM Throughput', 'Memory Throughput', 'Compute (SM) Throughput', 'Achieved Occupancy',
            'Registers Per Thread', 'Executed Ipc Active', 'Issue Slots Busy', 'Warp Cycles Per Issued Instruction']
    sel = d[d['Metric Name'].isin(want)]
    table = sel.pivot_table(index='Metric Name', columns='variant', values='Metric Value', aggfunc='first')
    units = sel.groupby('Metric Name')['Metric Unit'].first()
    table.insert(0, 'unit', units)
    print(table.reindex([w for w in want if w in table.index])[['unit'] + [v for v in order if v in table.columns]].to_string())

    raw = subprocess.run([NCU, '--import', rep + '.ncu-rep', '--csv', '--page', 'raw'], capture_output=True, text=True).stdout
    open(f'{OUT}/kernels_raw.csv', 'w').write(raw)
    rw = pd.read_csv(f'{OUT}/kernels_raw.csv', skiprows=[1], low_memory=False)
    stall_cols = [c for c in rw.columns if 'issue_stalled' in c and c.endswith('per_warp_active.pct')]
    pipe_cols = [c for c in rw.columns if re.search(r'pipe_(xu|fma|alu|fp64|lsu).*pct_of_peak_sustained_active$', c)]
    if stall_cols or pipe_cols:
        rw['variant'] = [order[k] if k < len(order) else f'launch {k}' for k in range(len(rw))]
        show = rw.set_index('variant')[pipe_cols + stall_cols].apply(pd.to_numeric, errors='coerce').T
        show = show[show.max(axis=1) > 2]           # keep the pipes / stall reasons that matter
        print('\nPipe utilization and warp stall reasons (% , rows > 2% somewhere)')
        print(show.round(1).to_string())
else:
    print('Part A skipped (no ncu or no counter permission). Part B below needs neither.')

**Reading Part A.** For vLLM's pass with random numbers, expect low `DRAM Throughput` (~20% of peak, as notebook 08's bandwidth suggested) with high compute / `pipe_xu` (the unit that computes `log` and `exp`) and stalls on math pipes; without random numbers, DRAM throughput near peak. Our fused top-k pass without random numbers should sit in between (its running top-k adds work).

## Part B — noise ablation (no counters needed)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
r = subprocess.run([sys.executable, 'profile_kernels.py', '--ablation', '--batch', str(BATCH), '--out', f'{OUT}/launch_order.json'],
                   capture_output=True, text=True, env=dict(os.environ))
print(r.stdout[-3000:]); print(r.stderr[-1500:])
abl = json.load(open(f'{OUT}/launch_order_ablation.json'))
ab = pd.DataFrame(abl['rows'])
base = float(ab.loc[ab['mode'] == 0, 'ms'].iloc[0])
ab['extra ms vs no noise'] = ab.ms - base
print(ab[['variant', 'ms', 'GBps', 'extra ms vs no noise']].to_string(index=False, float_format='{:.2f}'.format))

**Reading Part B.** `random numbers + 2 logs` minus `no noise` is the total cost of Gumbel-max sampling. Comparing `random numbers only` and `2 logs of a cheap hash` shows how that splits between the random-number generator and the logarithms. A pass close to ~1,500 GB/s is limited by memory; well below it, by computation.

## Part C — vLLM's built-in profiler on real traffic (needs the model, 2+ GPUs)

For each configuration: start a server with `--profiler-config` (vLLM's PyTorch profiler), send a few warm-up requests, call `/start_profile`, send `PROFILE_REQUESTS` GSM8K questions at `PROFILE_CONCURRENCY`, call `/stop_profile`, stop the server. Then GPU kernels in the trace are grouped into categories and divided by the number of denoising steps (counted from vLLM's per-step `_sample_step` kernel launches).

Set `RUN_PART_C = True` when two GPUs with ~36 GB free each are available. If the server rejects `--profiler-config`, check `vllm serve --help | grep -i profil` for this vLLM version's flag name.

In [ ]:
RUN_PART_C = os.environ.get('NB_RUN_PART_C', '0') == '1'
N_GPUS = int(os.environ.get('NB_N_GPUS', 2))
PROFILE_CONCURRENCY = 8
PROFILE_REQUESTS = 24
MODEL_ID = 'google/diffusiongemma-26B-A4B-it'
PORT = 8001
CONFIGS = {
    'stock':        {},
    'topk64':       {'DG_TOPK_SC_K': '64', 'DG_TOPK_SC_TAIL': 'renorm'},
    'topk64_exact': {'DG_TOPK_SC_K': '64', 'DG_TOPK_SC_TAIL': 'renorm', 'DG_TOPK_SAMPLE': 'topk_exact'},
}
CATEGORIES = [                          # first match wins; names of vLLM / PyTorch / Triton kernels
    ('sampler: statistics pass', r'_row_stats'),
    ('sampler: decisions (commit/stop)', r'_sample_step'),
    ('sampler: soft-cap', r'_softcap'),
    ('top-k (torch.topk)', r'gatherTopK|radixSort|topk'),
    ('hint: embedding_bag', r'embedding_bag|EmbeddingBag'),
    ('communication', r'nccl|all_reduce|allreduce|cross_device'),
    ('attention', r'attn|attention|flash|unified_'),
    ('MoE', r'moe|expert|fused_moe|marlin'),
    ('matmul (GEMM)', r'gemm|cutlass|xmma|ampere_|sm80_|cublas|Kernel2|matmul'),
]

In [ ]:
import asyncio, signal, urllib.request
from openai import AsyncOpenAI
from datasets import load_dataset

def pick_gpus(n):
    rows = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.free,memory.total', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()
    free = {int(i): (int(f) / 1024, int(t) / 1024) for i, f, t in (r.split(', ') for r in rows)}
    g = sorted(free, key=lambda k: -free[k][0])[:n]
    util = round(min(0.92, (min(free[k][0] for k in g) - 1.0) / free[g[0]][1]), 2)
    return ','.join(map(str, g)), util

def start_profiled_server(name, extra_env, gpus, util, trace_dir, timeout_s=1500):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, **extra_env)
    for key in ('DG_TOPK_SC_K', 'DG_TOPK_SC_TAIL', 'DG_TOPK_SAMPLE'):
        if key not in extra_env:
            env.pop(key, None)
    vllm_bin = shutil.which('vllm') or os.path.join(os.path.dirname(sys.executable), 'vllm')
    cmd = [vllm_bin, 'serve', MODEL_ID, '--port', str(PORT), '--max-num-seqs', str(PROFILE_CONCURRENCY),
           '--tensor-parallel-size', str(N_GPUS), '--max-model-len', '4096', '--attention-backend', 'TRITON_ATTN',
           '--enforce-eager', '--gpu-memory-utilization', str(util),
           '--profiler-config', json.dumps({'profiler': 'torch', 'torch_profiler_dir': os.path.abspath(trace_dir)})]
    log_path = f'{OUT}/server_profile_{name}.log'
    proc = subprocess.Popen(cmd, env=env, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            return None, log_path
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=5) as r:
                if r.status == 200:
                    return proc, log_path
        except Exception:
            pass
        time.sleep(10)
    os.killpg(proc.pid, signal.SIGKILL)
    return None, log_path

def stop(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM); proc.wait(timeout=120)
    except Exception:
        os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(15)

def post(path):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}{path}', method='POST', data=b'{}',
                                 headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=600) as r:
        return r.status

gsm = load_dataset('openai/gsm8k', 'main', split=f'test[:{PROFILE_REQUESTS + 4}]')
async def send(questions, conc):
    client = AsyncOpenAI(base_url=f'http://127.0.0.1:{PORT}/v1', api_key='none')
    sem = asyncio.Semaphore(conc)
    async def one(q):
        async with sem:
            await client.chat.completions.create(model=MODEL_ID, max_tokens=512, messages=[{'role': 'user', 'content': q}])
    await asyncio.gather(*(one(q) for q in questions))

def load_trace_kernels(trace_dir):
    files = sorted(glob.glob(f'{trace_dir}/**/*.json*', recursive=True))
    per_file = []
    for f in files:
        data = json.load(gzip.open(f) if f.endswith('.gz') else open(f))
        ev = [e for e in data.get('traceEvents', []) if e.get('ph') == 'X' and e.get('cat') in ('kernel', 'gpu_memcpy', 'gpu_memset')]
        per_file.append((f, ev))
    return per_file

def categorize(events):
    rows = {}
    for e in events:
        cat = next((c for c, pat in CATEGORIES if re.search(pat, e['name'], re.I)), 'other')
        rows[cat] = rows.get(cat, 0.0) + e['dur'] / 1000.0          # ms
    steps = sum(1 for e in events if re.search(r'_sample_step', e['name']))
    return rows, steps

In [ ]:
if RUN_PART_C:
    gpus, util = pick_gpus(N_GPUS)
    print('Part C on GPUs', gpus, '| --gpu-memory-utilization', util)
    summary = {}
    for name, extra in CONFIGS.items():
        trace_dir = f'{OUT}/trace_{name}'
        shutil.rmtree(trace_dir, ignore_errors=True); os.makedirs(trace_dir)
        print(f'{name}: starting profiled server')
        proc, log_path = start_profiled_server(name, extra, gpus, util, trace_dir)
        if proc is None:
            print('  failed to start:\n' + ''.join(open(log_path).readlines()[-10:])); continue
        try:
            await send([ex['question'] for ex in gsm.select(range(4))], PROFILE_CONCURRENCY)        # warm-up
            print('  start_profile:', post('/start_profile'))
            t0 = time.time()
            await send([ex['question'] for ex in gsm.select(range(4, 4 + PROFILE_REQUESTS))], PROFILE_CONCURRENCY)
            wall = time.time() - t0
            print('  stop_profile:', post('/stop_profile'), f'| profiled window {wall:.1f}s')
            time.sleep(20)                                         # let the workers finish writing traces
        finally:
            stop(proc)
        per_file = load_trace_kernels(trace_dir)
        if not per_file:
            print('  no trace files found in', trace_dir); continue
        f0, ev0 = per_file[0]                                      # one rank is enough: the ranks run in lockstep
        cats, steps = categorize(ev0)
        summary[name] = dict(steps=steps, **{k: v / max(steps, 1) for k, v in cats.items()})
        print(f'  {len(per_file)} trace file(s); rank file {os.path.basename(f0)}: {steps} sampler steps')
    if summary:
        tbl = pd.DataFrame(summary).T
        cols = [c for c, _ in CATEGORIES] + ['other']
        tbl = tbl[['steps'] + [c for c in cols if c in tbl]]
        tbl['total ms / step'] = tbl[[c for c in cols if c in tbl]].sum(axis=1)
        samp = [c for c in tbl if c.startswith(('sampler', 'top-k', 'hint'))]
        tbl['sampler kernels %'] = 100 * tbl[samp].sum(axis=1) / tbl['total ms / step']
        tbl.to_csv(f'{OUT}/step_breakdown.csv')
        print('\nGPU time per denoising step by category (ms), one tensor-parallel rank')
        print(tbl.round(2).T.to_string())
else:
    print('Part C is off (RUN_PART_C = False).')

**Reading Part C.**
* `total ms / step` is the GPU time of one denoising step (all requests in the batch together). Stock vs. ours is the end-to-end effect on step time at this concurrency.
* The full self-conditioning matmul is a cuBLAS GEMM, so in stock vLLM it is inside `matmul (GEMM)`: the drop in that row from `stock` to `topk64` is the hint matmul that our plugin removes; `hint: embedding_bag` is what replaces it.
* `sampler kernels %` is the share of the step in the kernels we can name (statistics pass, decisions, soft-cap, top-k, hint), so it understates stock's sampler share by the hint GEMM. Add the GEMM drop to see the full sampler share.
* Steps are counted from `_sample_step` launches, one per request tile per step, so with several tiles per step the absolute ms/step are scaled; compare configurations at the same concurrency.